In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("ai4i2020.csv")
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


UDI ve product id sadece kimlik verileri, makinanın arızalanması hakkında bilgi vermiyor.
TWF	HDF	PWF	OSF	RNF bunlar makinanın failure olup olmadığını gösteriyor. Modeli eğitirken bunları feauture olarak veremeyiz data leakage olur.
Bu yüzden bahsettiğimiz bu özellikleri drop ediyoruz aşağıda.

In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  object 
 2   Type                     10000 non-null  object 
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9)

In [ ]:
df["Machine failure"].value_counts()

,count
Machine failure,
0,9661
1,339


In [ ]:
df = df.drop(["UDI","Product ID","TWF","HDF","PWF","OSF","RNF"], axis=1)
#axis=1 => sütunu sil
#ONE HOT ENCODING  type  metinleri=>sayısala çevirme (0/1)
df = pd.get_dummies(df, columns=["Type"], drop_first=True)

df.head()

,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,Type_L,Type_M
0,298.1,308.6,1551,42.8,0,0,False,True
1,298.2,308.7,1408,46.3,3,0,True,False
2,298.1,308.5,1498,49.4,5,0,True,False
3,298.2,308.6,1433,39.5,7,0,True,False
4,298.2,308.7,1408,40.0,9,0,True,False


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler

In [ ]:
#input (x) ve output (y) ayırıyoruz
x = df.drop("Machine failure",axis=1).values
y = df["Machine failure"].values

#%60 train sonra kalan %40 => %20 validaiton ve test set
x_train, x_temp, y_train, y_temp = train_test_split(
    x, y, test_size=0.4, random_state=42, stratify=y
)
x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


stratify => her parçada arıza oranı aynı kalsın
random_state=42 => bölmeye her çalıştığımda aynı yap 42 geleneksel bir sayı.

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y) *italik metin*

Soldakiler sırasıyla input1, input2, target1,target2
test_size=0.4 ile verinin >%40'ını ayırdık &60'ını traine verdik.
normalde 3 parçaya ayırmamız gerekli ama train_test_split bir seferde yalnızca 2 parçaya ayırabilir.
burada x_test değil x_temp dedik çünkü onu da ikiye böleceğiz.


In [ ]:
#Scaling (fit sadece trainde yapılır)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)
x_test = scaler.transform(x_test)

In [ ]:
#Oversampling (sadece trainde yapılır)
ros = RandomOverSampler(random_state=42)
x_train,y_train = ros.fit_resample(x_train,y_train)

print("Train dağılımı:", np.bincount(y_train))
print("Validation boyutu:", len(y_val), "|Test Boyutu:", len(y_test))

Train dağılımı: [5797 5797]
Validation boyutu: 2000 |Test Boyutu: 2000


fit_transform=> veriden mean/std öğrenir fit ile, sonra aynı veriyi hemen o değerlere dönüştürür (transform). Sadece trainde kullanılır.

transform => yeniden öğrenmez, trainde öğrenilen mean/std alıp val ve teste uygular.

fit_resample => azınlık sınıfı çoğaltıp iki sınıfı dengeler. (sadece trainde)

np.bicount=> bir dizideki her tam sayının kaç kez geçtiğini sayar. bizim datasetimizde kaç sağlam kaç hatalı olduğunu gösteriyor.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
#Baseline model : Logistic Regression
log_model = LogisticRegression()
log_model.fit(x_train,y_train)

#validation setinde tahmin
y_pred = log_model.predict(x_val)

print(classification_report(y_val,y_pred))
print(confusion_matrix(y_val,y_pred))

              precision    recall  f1-score   support

           0       0.99      0.84      0.91      1932
           1       0.15      0.82      0.26        68

    accuracy                           0.84      2000
   macro avg       0.57      0.83      0.58      2000
weighted avg       0.96      0.84      0.89      2000

[[1621  311]
 [  12   56]]


Imbalanced veride accuracy yalan söyler. Model hep sağlam dese %98 doğruluk alır mesela ama tek arızayı yakalayamaz. Bu yüzden özellikle arıza=1 sınıfı için şu özelliklere bakmamız gerekli:

**Precision => **model arızalanacak dediklerinin % kaçı gerçekten arızalandı? Yüksek precision = az yanlış alarm

**Recall=>** Gerçekten arızalanan makinelerin % kaçını model yakaladı? Yüksek recall= az kaçırılan kaza

**F1-score =>** Precision ve recall'un dengesi

Özwellike bu datasette recall daha önemlidri. Bir false alarm can sıkar ama ucuzdur ama missed defects ise makineyi patlatabilir.

Tabloyu yorumlarsak:

**support =>** validation'da o sınıftan kaç gerçek örnek var

mesela 1=>arızalıda  **recall=0.82** demek gerçekten arızalı 68 makinenin %82'sini yakaladı

**precision=0.15** arızalanacak dediklerinin sadece %15'i gerçekten arızalandı yani çok fazla false alarm var

sondaki confusion matrix [[1621  311]
 [  12   56]]
 Tahmin:0   Tahmin:1
Gerçek:0  [  1621        311   ]   ← 1932 sağlam makine
Gerçek:1  [    12         56   ]   ←   68 arızalı makine


özetle
1621=> sağlamı doğru bildi
311 => sağlama arızalı dedi **FALSE ALARM**
12=> arızayı kaçırdı **Missed defects tehlikeli**
56=> arızayı doğru yakaladı

train'i oversampling ile dengelediğimiz için model "arıza" demeye meyilli hale geldi. Yüksek recall iyi ama bunu 311 false alarm pahasına düşük precision ile yapıyor. Çok arıza yakala ama çok da false alarm ver.

**macro avg** = (class0 precision + class1 precision) /2
sınıfları eşit sayan, küçük ama önemli arıza sınıfı görünür kalır.
Dengesiz veride dürüst ölçü verir.

**weighted avg =** örnek sayısına göre mesela bizim datasetimizde
(0.99 * 1932 + 0.15*68) /2000
örnek sayısına göre ağırlıklı, çoğunluk sınıf baskındır (dengesiz veride yanıltıcıdır)



#KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(x_train, y_train)

y_pred_knn = knn_model.predict(x_val)
print("*******KNN**********")
print(classification_report(y_val,y_pred_knn))
print(confusion_matrix(y_val, y_pred_knn))

*******KNN**********
              precision    recall  f1-score   support

           0       0.99      0.96      0.97      1932
           1       0.36      0.62      0.45        68

    accuracy                           0.95      2000
   macro avg       0.67      0.79      0.71      2000
weighted avg       0.96      0.95      0.96      2000

[[1856   76]
 [  26   42]]


#NNN with Keras

In [ ]:
import tensorflow as tf

nn_model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(x_train.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

nn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
nn_model.fit(x_train, y_train, epochs=20, batch_size=32, verbose=0)


#predict 0.5 eşiği ile 0/1'a çevirerek olasılık verir
y_pred_nn = (nn_model.predict(x_val)>0.5).astype(int)
print("*****Neural Network")
print(classification_report(y_val, y_pred_nn))
print(confusion_matrix(y_val, y_pred_nn))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
*****Neural Network
              precision    recall  f1-score   support

           0       0.99      0.93      0.96      1932
           1       0.31      0.85      0.46        68

    accuracy                           0.93      2000
   macro avg       0.65      0.89      0.71      2000
weighted avg       0.97      0.93      0.95      2000

[[1805  127]
 [  10   58]]


Özellike NN kısmmı için notlarım:
#Model kurma

**Sequential =** Katmanları sırayla (alt alta) dizen model yapısı; veri üstten girer, aşağı doğru akar.

**Dense(32) =** 32 nöronlu "tam bağlı" katman; her nöron bir önceki katmandaki tüm nöronlara bağlıdır.

**activation='relu' =** Nöronun çıkışına uygulanan fonksiyon; ReLU negatifi 0 yapar, pozitifi geçirir. Gizli katmanlarda kullanılır, ağa nonlinearity katar.

**activation='sigmoid' = **Çıkışı 0–1 arasına sıkıştırır (olasılık üretir); binary classificaiton çıkış katmanında kullanılır.

**input_shape=(x_train.shape[1],) =** Ağa "girdim kaç özellikten oluşuyor" bilgisini verir. x_train.shape[1] = sütun (özellik) sayısı = bizde 7.

#Compile => modele nasıl öğreneceğini söylemek

**compile = **Modeli eğitime hazırlama adımı; üç şeyi belirler: optimizer, loss, metrics.

**optimizer =** Ağırlıkları hatayı azaltacak yönde güncelleyen algoritma (öğrenme yöntemi).

**Adam =** En yaygın optimizer; gelişmiş bir gradient descent (yokuş aşağı inme) türü, hızlı ve stabil.

**loss (kayıp) = **Modelin ne kadar yanlış olduğunu ölçen sayı; eğitim bunu küçültmeye çalışır.

**binary_crossentropy =** Binary classificationın (0/1) loss fonksiyonu. (Regresyonda bunun yerine MSE kullanılır.)

**metrics =** Eğitim sırasında bizim izlediğimiz başarı ölçüsü; loss'tan farkı, modelin optimize ettiği değil, insanın takip ettiği değer olması.

**metrics=['accuracy'] =** Keras'ın izleyip yazdırdığı ölçü olarak doğruluğu seç. (Not: dengesiz veride accuracy yanıltıcı olduğu için asıl kararı yine classification_report'taki precision/recall ile veririz; buradaki accuracy sadece eğitimi izlemek için.)

#Training ve Prediction
**fit =** Modeli eğitim verisiyle eğit (ağırlıkları öğren).

**epochs=20 = **Modelin tüm eğitim verisini kaç kez baştan sona göreceği (20 tur).

**batch_size=32 = **Veriyi 32'lik gruplar halinde işle; her 32 örnekte bir ağırlıkları güncelle.

**verbose=0 =** Eğitim çıktısını ekrana basma (sessiz mod).

**(predict > 0.5).astype(int) = **Sigmoid olasılığını 0.5 eşiğiyle k (0/1) çevir.


#Logistic Regressin vs KNN vs Neural Network

**LogReg**
F1 => 0.26
precision
0.15
recall
0.82
Yakalanan arıza oranı:
56/68
missed defects
12
False Alarm
311

**KNN**
F1 => 0.45
precision
0.36
recall
0.62
Yakalanan arıza oranı:
42/68
missed defects
26
False Alarm
76

**Neural Network**
F1 => 0.46
precision
0.31
recall
0.85
Yakalanan arıza oranı:
58/68
missed defects
10
False Alarm
127

KNN en temkinli olanı, en az false alarma sahip ve en yüksek precisiona sahip, yani arızalı dediğinde daha çok haklı ,ancak recall 0.62 26 arızayı kaçırıyor, bu çok riskli.

LogReg baseline aslında. False alarmı çok yüksek.

Neural networkü seçmeliyiz, çünkü arızaların %85'ini yakalıyor ve sadece 10 tanesini kaçırdı. Missed defect maliyetleri false alarma göre hem daha pahalıdır hem de üretim süreçlerinde tehlikeli sonuçları olabilir. Bu yüzden neural networkü seçiyorum.





In [ ]:

y_pred_test = (nn_model.predict(x_test) > 0.5).astype(int)

print("***** TEST SETİ — Neural Network *****")
print(classification_report(y_test, y_pred_test))
print(confusion_matrix(y_test, y_pred_test))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
***** TEST SETİ — Neural Network *****
              precision    recall  f1-score   support

           0       1.00      0.94      0.97      1932
           1       0.34      0.88      0.49        68

    accuracy                           0.94      2000
   macro avg       0.67      0.91      0.73      2000
weighted avg       0.97      0.94      0.95      2000

[[1815  117]
 [   8   60]]


Test setinde model, gerçek arızalarının %88'ini 68'den sadece 8'ini kaçırdı.